# Grounded RAG Fine-Tuning with Unsloth (QLoRA) using Llama-3 Chat Template

This notebook fine-tunes `unsloth/llama-3-8b-Instruct-bnb-4bit` to strictly follow context-grounding instructions for RAG systems.
Key features:
- Native Meta Llama-3 Chat Template (`<|begin_of_text|><|start_header_id|>...`) via Unsloth `get_chat_template`
- 4-bit QLoRA with Unsloth fast backpropagation kernels
- Target modules: `q_proj`, `k_proj`, `v_proj`, `o_proj`, `gate_proj`, `up_proj`, `down_proj` (r=16)
- Supervised fine-tuning (`SFTTrainer`) on context-grounded RAG dataset for 1 epoch
- Direct 4-bit GGUF (`q4_k_m`) export for deployment with Ollama

### 1. Environment Setup and Dependency Installation

In [ ]:
%%capture
# Install Unsloth and required fine-tuning libraries
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes datasets

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

### 2. Load Pretrained 4-bit Llama-3-8B-Instruct

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None  # None for auto-detection (Float16 or Bfloat16)
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

### 3. Attach QLoRA Adapters & Apply Native Llama-3 Chat Template

In [ ]:
# 1. Configure QLoRA PEFT model
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,  # Optimized = 0 in Unsloth
    bias="none",     # Optimized = "none" in Unsloth
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

# 2. Apply Unsloth native Llama-3 chat template to tokenizer
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template="llama-3",  # Sets up <|begin_of_text|><|start_header_id|>system/user/assistant format
)

print("[✓] QLoRA adapter attached and native Llama-3 chat template configured.")

### 4. Load Dataset and Format using Native Llama-3 Chat Template

In [ ]:
from datasets import load_dataset

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs = examples["input"]
    outputs = examples["output"]
    texts = []
    for instruction, input_text, output in zip(instructions, inputs, outputs):
        # Structure as standard multi-turn conversation (system, user, assistant)
        messages = [
            {"role": "system", "content": instruction},
            {"role": "user", "content": input_text},
            {"role": "assistant", "content": output},
        ]
        # Use native Llama-3 tokenizer chat template
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
        texts.append(text)
    return {"text": texts}

dataset = load_dataset("json", data_files="dataset.json", split="train")
dataset = dataset.map(formatting_prompts_func, batched=True)

print(f"Loaded {len(dataset)} training examples.")
print("\n--- Sample Formatted Llama-3 Prompt ---")
print(dataset[0]["text"][:350] + "...")

### 5. Configure and Execute Training with SFTTrainer (1 Epoch)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=1,  # Trained for 1 epoch
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)

trainer_stats = trainer.train()
print("[✓] Training completed.")

### 6. Verify Context-Grounding In-Memory using Llama-3 Chat Template

In [ ]:
FastLanguageModel.for_inference(model)  # 2x inference speedup

test_messages = [
    {
        "role": "system",
        "content": "Answer the user query based ONLY on the provided context. If the answer cannot be determined strictly from the context, reply with: \"Information not found.\""
    },
    {
        "role": "user",
        "content": "Context:\nProject Titan uses PostgreSQL 16 as its primary relational store.\n\nQuestion:\nWhat database is used for caching?"
    },
]

prompt = tokenizer.apply_chat_template(
    test_messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=32, use_cache=True)
response = tokenizer.batch_decode(outputs)
print("Model Raw Output:")
print(response[0].split("<|start_header_id|>assistant<|end_header_id|>")[-1].strip())

### 7. Export Merged Model to 4-bit GGUF (`q4_k_m`)

Unsloth merges LoRA weights into the base model and quantizes to 4-bit `q4_k_m` GGUF for Ollama.

In [ ]:
# Export to 4-bit GGUF (q4_k_m)
model.save_pretrained_gguf(
    "rag_grounded_llama3",
    tokenizer,
    quantization_method="q4_k_m"
)
print("[✓] Exported model in 4-bit GGUF (q4_k_m) successfully!")